# p225 Implement Stack using Queues 解析笔记

- 题号：p225
- 题目英文名：Implement Stack using Queues
- 题目中文名：用队列实现栈
- 题目链接：https://leetcode.cn/problems/implement-stack-using-queues/
- 题型：栈与队列设计
- 难度：Easy
- 推荐优先级：低
- Java 原代码位置：`solutions-java/leetcode/p225_implement_stack_using_queues/ImplementStackUsingQueues.java`


## 1. 题目一句话总结

这道题要求我们只用队列的合法操作去模拟一个后进先出的栈。

本质上考的是如何用先进先出的结构，通过重排操作制造出后进先出的访问效果。


## 2. 题目理解与约束分析

### 2.1 题目要求

实现一个 `MyStack`，支持：

- `push(x)`
- `pop()`
- `top()`
- `empty()`

但底层只能使用队列的标准操作。

### 2.2 输入与输出

- 输入：一系列栈操作
- 输出：对应的 `pop/top/empty` 结果
- 返回结果含义：行为必须与真实栈一致

### 2.3 关键约束

- 只能使用队列接口
- 栈要求后进先出
- 队列天然是先进先出，所以必须人为重排

### 2.4 示例分析

如果依次执行：

```text
push(1)
push(2)
top() -> 2
pop() -> 2
empty() -> false
```

说明最新压入的元素必须最先被取出。


## 3. Java 原代码

```java
package leetcode.p225_implement_stack_using_queues;

import java.util.LinkedList;
import java.util.Queue;

public class ImplementStackUsingQueues {

    public static class MyStack {
        private Queue<Integer> queue;

        public MyStack() {
            queue = new LinkedList<>();
        }

        public void push(int x) {
            int n = queue.size();
            queue.offer(x);
            for (int i = 0; i < n; i++) {
                queue.offer(queue.poll());
            }
        }

        public int pop() {
            return queue.poll();
        }

        public int top() {
            return queue.peek();
        }

        public boolean empty() {
            return queue.isEmpty();
        }
    }
}
```


## 4. 先从 Java 原方案理解

Java 原方案只用一个队列就完成了栈模拟，关键在 `push` 这一步的旋转操作。

它的思路是：

1. 先把新元素 `x` 放到队尾
2. 再把原来队列里的前 `n` 个元素依次出队、入队
3. 这样新元素就被转到了队头

一旦新元素被放到了队头：

- `pop()` 直接弹队头
- `top()` 直接看队头

于是队列的队头就始终对应栈顶。


## 5. 从朴素思路到优化思路

### 5.1 最直接的想法

如果不限制结构，直接用栈当然最简单。

### 5.2 为什么这里难

因为队列是先进先出，而栈是后进先出，访问方向天然相反。

### 5.3 优化方向

Java 原方案的关键想法是：既然不能改变队列接口，就在 `push` 时主动重排，让刚压入的元素立刻来到队头。


## 6. 核心算法讲解

### 6.1 算法名称

- 单队列模拟栈
- 队列旋转

### 6.2 为什么想到这个算法

因为如果我们能保证“最新元素永远在队头”，那队列的队头就能直接充当栈顶。

### 6.3 关键状态

- `queue`：唯一的底层队列
- `n`：压入新元素前的队列长度

### 6.4 步骤拆解

1. `push(x)` 时先记录当前长度 `n`
2. 把 `x` 放到队尾
3. 把前面 `n` 个旧元素依次搬到队尾
4. 此时 `x` 就到了队头
5. `pop()` 直接取队头
6. `top()` 直接看队头


## 7. 过程演示

依次执行：`push(1), push(2), push(3)`

```text
初始：queue = []

push(1)：
先放入 1 -> [1]
旧元素个数 n = 0，不需要旋转

push(2)：
先放入 2 -> [1, 2]
旋转 1 次：poll 1 再 offer 1 -> [2, 1]

push(3)：
先放入 3 -> [2, 1, 3]
旋转 2 次：
[1, 3, 2]
[3, 2, 1]
```

现在队头就是 3，也就是栈顶。


In [ ]:
from collections import deque


class MyStack:
    def __init__(self) -> None:
        self.queue: deque[int] = deque()

    def push(self, x: int) -> None:
        n = len(self.queue)
        self.queue.append(x)
        for _ in range(n):
            self.queue.append(self.queue.popleft())

    def pop(self) -> int:
        return self.queue.popleft()

    def top(self) -> int:
        return self.queue[0]

    def empty(self) -> bool:
        return not self.queue


## 8. 代码逐段讲解

### 8.1 `push`

这一步完全对齐 Java 原解。真正让栈成立的不是 `append`，而是后面的那一轮旋转。

### 8.2 `pop` 和 `top`

因为最新元素已经在队头，所以：

- `pop()` 直接弹队头
- `top()` 直接看队头

### 8.3 为什么这版是“把复杂度放在 push 上”

Java 原方案把重排成本都放到了 `push`，于是 `pop/top` 就非常轻。


## 9. 复杂度分析

- `push()` 时间复杂度：`O(n)`
- 为什么是这个时间复杂度：每次压入后都要旋转已有元素
- `pop()`、`top()`、`empty()` 时间复杂度：`O(1)`
- 空间复杂度：`O(n)`
- 为什么是这个空间复杂度：所有元素都存放在队列中


## 10. 易错点

1. `push` 后忘了旋转旧元素，结果队头不是栈顶。
2. 旋转次数写错，不是旋转 `n` 次而是旋转了错误数量。
3. 误以为只靠一个队列不能模拟栈，其实 Java 原方案已经证明可以。
4. 把复杂度理解反了，这版是 `push` 慢、`pop/top` 快。


## 11. 面试时怎么讲

可以这样概括：

> 这题我会沿着 Java 原方案，用一个队列模拟栈。
> 做法是在每次 `push` 时先把新元素放到队尾，再把前面的旧元素全部依次搬到后面。
> 这样新元素就会来到队头，于是队头始终对应栈顶。
> 所以 `pop` 和 `top` 可以直接操作队头，代价主要集中在 `push` 上。


## 12. 实际应用场景

这题可以类比成：底层只有一种顺序容器接口，但上层想伪装出另一种访问语义。

### 具体业务案例：消息通道模拟撤销栈

#### 业务背景

某些系统底层只有队列式消息通道，但上层业务想实现“最近操作先撤销”。

#### 输入是什么

输入是一系列推入、弹出、查看顶部操作。

#### 算法介入点

系统通过在入队时重排顺序，把队列改造成栈的行为。

#### 输出是什么

输出是符合后进先出语义的结果。

#### 价值是什么

它解决的是“接口受限但语义要适配”的设计问题。


In [ ]:
stack = MyStack()
stack.push(1)
stack.push(2)
stack.push(3)
print('当前栈顶:', stack.top())
print('弹出:', stack.pop())
print('再次查看栈顶:', stack.top())
print('是否为空:', stack.empty())


## 13. Demo 输出说明

- 第一行应输出 `3`，说明最新压入元素已经被旋转到队头。
- 第二行应输出 `3`，说明弹出顺序符合栈的后进先出。
- 第三行应输出 `2`，说明剩余栈顶更新正确。


## 14. 一句话复盘

> 这题最关键的不是用队列，而是像 Java 原解那样把新元素主动旋转到队头。
